# BrowserGym Runner fuer GitLab Task 44

Dieses Notebook ist der Schritt nach dem direkten Playwright-Minimalrunner. Es nutzt BrowserGym fuer Browser-Environment, Observation und Action-Ausfuehrung, schreibt aber weiterhin die WebArena-Verified-Artefakte:

- `agent_response.json`
- `network.har`
- danach `eval_result.json` durch WebArena-Verified

In [1]:
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OFFICIAL_REPO = PROJECT_ROOT / 'external' / 'webarena-verified'
OFFICIAL_REPO

PosixPath('/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified')

## 1. Demo-GitLab muss laufen

Falls nicht:

```bash
cd external/webarena-verified
uv run invoke -r examples gitlab-start
```

In [2]:
subprocess.run(['docker', 'ps', '--filter', 'name=wa-demo-gitlab'], check=True)

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


CompletedProcess(args=['docker', 'ps', '--filter', 'name=wa-demo-gitlab'], returncode=0)

## 2. Task-Input erzeugen

In [3]:
(OFFICIAL_REPO / 'output').mkdir(exist_ok=True)
subprocess.run([
    'uv', 'run', 'webarena-verified', 'agent-input-get',
    '--task-ids', '44',
    '--config', 'examples/configs/config.demo.json',
    '--output', 'output/tasks.demo.json',
], cwd=OFFICIAL_REPO, check=True)

json.loads((OFFICIAL_REPO / 'output/tasks.demo.json').read_text())

[WebArena Verified] [INFO] Loading config from: '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/examples/configs/config.demo.json'
[WebArena Verified] [INFO] Using test_data_file: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json
[WebArena Verified] [INFO] No config provided, using default configuration
[WebArena Verified] [INFO] Using test_data_file: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json
[WebArena Verified] [INFO] Loading tasks from: '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json'
[WebArena Verified] [INFO] Loaded 812 tasks successfully.
[WebArena Verified] [INFO] WebArenaVerified initialized successfully
[WebArena Verified] [INFO] Wrote 1 agent inputs to output/tasks.demo.json


[{'sites': ['gitlab'],
  'task_id': 44,
  'intent_template_id': 303,
  'start_urls': ['http://localhost:8012'],
  'intent': 'Open my todos page'}]

## 3. BrowserGym-Runner starten

Der Runner benutzt `browsergym/openended`, loggt sich in GitLab ein, fuehrt eine BrowserGym-Action `goto(...)` aus und evaluiert danach.

In [4]:
subprocess.run([
    str(PROJECT_ROOT / '.venv/bin/python'),
    str(PROJECT_ROOT / 'scripts/run_browsergym_gitlab_task44_runner.py'),
    '--repo-root', str(OFFICIAL_REPO),
    '--tasks-file', 'output/tasks.demo.json',
    '--task-id', '44',
    '--output-root', 'output/browsergym-run',
    '--config', 'examples/configs/config.demo.json',
], cwd=PROJECT_ROOT, check=True)

BrowserGym task 44:  67%|██████▋   | 4/6 [00:09<00:05,  2.65s/step]


BrowserGym observation keys
['active_page_index', 'axtree_object', 'chat_messages', 'dom_object', 'elapsed_time', 'extra_element_properties', 'focused_element_bid', 'goal', 'goal_object', 'last_action', 'last_action_error', 'open_pages_titles', 'open_pages_urls', 'screenshot', 'url']

Artifacts
- task_output_dir: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/browsergym-run/44
- network.har: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/browsergym-run/44/network.har (15896974 bytes)
- agent_response.json: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/browsergym-run/44/agent_response.json (104 bytes)

Evaluation stdout
[WebArena Verified] [INFO] Using config from --config: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/examples/configs/config.demo.json
[WebArena Verified] [INFO]

BrowserGym task 44: 100%|██████████| 6/6 [00:10<00:00,  1.73s/step]


CompletedProcess(args=['/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/.venv/bin/python', '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/scripts/run_browsergym_gitlab_task44_runner.py', '--repo-root', '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified', '--tasks-file', 'output/tasks.demo.json', '--task-id', '44', '--output-root', 'output/browsergym-run', '--config', 'examples/configs/config.demo.json'], returncode=0)

## 4. Ergebnis inspizieren

In [5]:
run_dir = OFFICIAL_REPO / 'output/browsergym-run/44'
sorted(p.name for p in run_dir.iterdir())

['agent_response.json', 'eval_result.json', 'network.har']

In [6]:
json.loads((run_dir / 'agent_response.json').read_text())

{'task_type': 'NAVIGATE',
 'status': 'SUCCESS',
 'retrieved_data': None,
 'error_details': None}

In [7]:
eval_result = json.loads((run_dir / 'eval_result.json').read_text())
{key: eval_result.get(key) for key in ['task_id', 'status', 'score']}

{'task_id': 44, 'status': 'success', 'score': 1.0}

## Einordnung

Wenn dieser Run `score = 1.0` erreicht, ist BrowserGym erfolgreich zwischen Task-Input und WebArena-Verified-Evaluation eingebaut. Danach ist AgentLab der naechste Runner-Layer: nicht mehr nur ein einzelner scripted Agent, sondern wiederholbare Experimente mit Result-Struktur.